# 01 · Raw to Bronze

Carrega o `fraudTrain.csv` do Cloud Storage para uma tabela gerenciada no
BigQuery. Espelho fiel do arquivo: campos de negócio todos como `STRING`, mais
dois metadados de governança.

O `fraudTest.csv` não entra aqui — permanece em `gs://BUCKET/holdout/` para o
Trabalho 2.

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.cloud import storage

In [ ]:
!pip install --quiet --upgrade google-cloud-bigquery google-cloud-storage db-dtypes


In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado no Colab — use a conta dona do projeto")
except ImportError:
    print("Fora do Colab: usando as credenciais do ambiente")

## Configuração

In [ ]:
PROJECT_ID = "fraudflow-pdm-gps"
BUCKET_NAME = "fraudflow-pdm-gps-data"
ARQUIVO_TRAIN = "raw/train/fraudTrain.csv"
ARQUIVO_HOLDOUT = "holdout/fraudTest.csv"

GCS_URI      = f"gs://{BUCKET_NAME}/{ARQUIVO_TRAIN}"
TABELA_STG    = f"{PROJECT_ID}.bronze.transactions_stg"
TABELA_BRONZE = f"{PROJECT_ID}.bronze.transactions"

client = bigquery.Client(project=PROJECT_ID)
gcs = storage.Client(project=PROJECT_ID)

print(f"Origem  : {GCS_URI}")
print(f"Destino : {TABELA_BRONZE}")

## Conferência do bucket

Somente metadados. O tamanho denuncia troca de lugar: o treino é o maior.

In [ ]:
objetos = {b.name: b.size for b in gcs.list_blobs(BUCKET_NAME) if b.name.endswith('.csv')}
for nome, tamanho in sorted(objetos.items()):
    print(f"  {nome:<34} {tamanho/1024**2:8.1f} MB")

assert ARQUIVO_TRAIN in objetos, f"faltando {ARQUIVO_TRAIN}"
assert ARQUIVO_HOLDOUT in objetos, f"faltando {ARQUIVO_HOLDOUT}"
assert objetos[ARQUIVO_TRAIN] > objetos[ARQUIVO_HOLDOUT], \
    "o treino deveria ser o MAIOR — train e holdout trocaram de lugar"
print("\nLayout ok: holdout separado")

## Schema do arquivo de origem

O CSV tem 23 colunas, mas a primeira não tem nome no cabeçalho — é o índice
gravado quando o dataset foi exportado. O schema é declarado explicitamente e
essa coluna recebe o nome `row_id`; com `skip_leading_rows=1` o BigQuery ignora
o cabeçalho e mapeia por posição.

As demais mantêm o nome original: a Bronze não renomeia nada.

In [ ]:
COLUNAS_RAW = [
    "row_id", "trans_date_trans_time", "cc_num", "merchant", "category", "amt",
    "first", "last", "gender", "street", "city", "state", "zip", "lat", "long",
    "city_pop", "job", "dob", "trans_num", "unix_time", "merch_lat",
    "merch_long", "is_fraud",
]

# Todos como STRING: e a regra da camada. Se o dado chegou torto, ele fica torto
# aqui, e a decisao de descartar ou corrigir e da Silver.
schema = [bigquery.SchemaField(c, 'STRING') for c in COLUNAS_RAW]
print(f"{len(schema)} colunas declaradas")

### Amostra do arquivo

`nrows` limitado a alguns megabytes, apenas para conferir o formato dos campos
antes de disparar a carga.

In [ ]:
# Le apenas os primeiros ~2 MB do objeto usando o MESMO cliente ja autenticado
# que funcionou no notebook 00. Evita a dependencia gcsfs, que exige credencial
# propria e e a falha mais comum ao ler gs:// pelo pandas dentro do Colab.
import io as _io

NL = chr(10)   # quebra de linha, sem escape

bucket = gcs.bucket(BUCKET_NAME)
trecho = bucket.blob(ARQUIVO_TRAIN).download_as_bytes(start=0, end=2_000_000)
texto  = trecho.decode('utf-8', errors='ignore')

# A ultima linha do trecho quase certamente veio cortada no meio: descarta.
linhas_csv = texto.splitlines()[:-1]

df_amostra = pd.read_csv(_io.StringIO(NL.join(linhas_csv)),
                         dtype=str, header=0, names=COLUNAS_RAW)

print(f'Amostra: {len(df_amostra)} linhas (dos primeiros 2 MB)')
print(f'Colunas conferem: {list(df_amostra.columns) == COLUNAS_RAW}')
print(f"Valores de is_fraud: {sorted(df_amostra['is_fraud'].unique())}")
print()
print('FORMATO DO TIMESTAMP  <-- anote, decide o notebook 02:')
print('   ', repr(df_amostra['trans_date_trans_time'].iloc[0]))
print("    esperado: '2019-01-01 00:00:18'   (padrao %Y-%m-%d %H:%M:%S)")
print()
print('FORMATO DO DOB:')
print('   ', repr(df_amostra['dob'].iloc[0]), "  esperado: '1988-03-09'")
df_amostra.head(3)

## Carga

`load_table_from_uri` é uma ordem, não uma transferência: o notebook envia
algumas centenas de bytes e o BigQuery faz a leitura paralelizada do Cloud
Storage. Nenhum dado trafega por aqui.

O job de carga não grava colunas calculadas, então ele escreve numa tabela de
estágio e a célula seguinte materializa a Bronze com os metadados.

In [ ]:
job_config = bigquery.LoadJobConfig(
    schema=schema,
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    field_delimiter=',',
    allow_quoted_newlines=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_uri(GCS_URI, TABELA_STG, job_config=job_config)
load_job.result()

print(f"Lidos do GCS : {load_job.input_file_bytes/1024**2:,.1f} MB")
print(f"Linhas        : {load_job.output_rows:,}")
print(f"Trafegou pelo Colab: 0 bytes de dado")

## Metadados de governança

`source_file` registra de qual arquivo cada linha veio, o que permite verificar
que o holdout ficou de fora. `ingestion_timestamp` torna a carga auditável.

Sem particionamento, de propósito: a carga acontece de uma vez só, então
particionar por data de ingestão criaria uma única partição.

In [ ]:
SQL_BRONZE = f"""
CREATE OR REPLACE TABLE `{TABELA_BRONZE}` AS
SELECT
  s.*,
  CURRENT_TIMESTAMP() AS ingestion_timestamp,
  '{GCS_URI}'         AS source_file
FROM `{TABELA_STG}` AS s
"""

job = client.query(SQL_BRONZE)
job.result()

client.delete_table(TABELA_STG, not_found_ok=True)
print(f"Bronze materializada · BigQuery leu {job.total_bytes_processed/1024**2:,.1f} MB")
print("Estagio removido")

## Validação

Tem que dar exatamente 1.296.675. Número menor indica cópia truncada: as versões
que circulam fora do Kaggle param em 1.048.575, o limite de linhas do Excel.

In [ ]:
tabela = client.get_table(TABELA_BRONZE)
print(f"Linhas   : {tabela.num_rows:,}")
print(f"Tamanho  : {tabela.num_bytes/1024**2:.1f} MB")
print(f"Colunas  : {len(tabela.schema)}")
print(f"Particao : {tabela.time_partitioning}")

assert tabela.num_rows == 1_296_675, f"esperado 1.296.675, veio {tabela.num_rows:,}"
assert tabela.time_partitioning is None, "a Bronze nao deve ser particionada"
print("\nBronze ok")

Confirmação de que só o treino entrou. Esta consulta agrega 1,3 milhão de linhas
no BigQuery e devolve **uma**:

In [ ]:
client.query(f"""
    SELECT source_file, COUNT(*) AS linhas, MIN(ingestion_timestamp) AS carregado_em
    FROM `{TABELA_BRONZE}`
    GROUP BY source_file
""").to_dataframe()

---

**Próximo:** `02_bronze_to_silver.ipynb`